
# SQLAlchemy Learning Notebook

This notebook is a practical guide to the **SQLAlchemy** Python library.

It covers:
- Jupyter setup
- database engines and connections
- Core vs ORM
- metadata, tables, columns, and constraints
- creating and dropping tables
- inserting, selecting, updating, and deleting data
- filtering, ordering, grouping, joins, and aggregates
- transactions
- sessions
- declarative ORM models
- relationships
- querying ORM objects
- common result helpers
- useful inspection helpers

**Note:** SQLAlchemy is a large library, so this notebook focuses on the **most commonly used methods, functions, and features** rather than every single API entry.

This notebook uses **SQLite** so it works locally without setting up a database server.



## 1) Jupyter setup

Install SQLAlchemy and Jupyter:

```bash
pip install sqlalchemy jupyter
jupyter notebook
```

If you want to use PostgreSQL later, you may also install a driver such as:

```bash
pip install psycopg[binary]
```

For MySQL, a common driver is:

```bash
pip install pymysql
```


In [2]:

from pathlib import Path

from sqlalchemy import (
    create_engine, MetaData, Table, Column,
    Integer, String, Float, Boolean, ForeignKey,
    DateTime, Text, select, insert, update, delete,
    func, and_, or_, not_, desc, asc, text
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column,
    Session, relationship
)
from datetime import datetime

print("SQLAlchemy imported successfully.")


SQLAlchemy imported successfully.


## 2) Create a local SQLite database

In [4]:

db_dir = Path("sqlalchemy_demo")
db_dir.mkdir(exist_ok=True)

db_path = db_dir / "demo.db"
engine = create_engine(f"sqlite:///{db_path}", echo=False)

print("Database path:", db_path.resolve())
print("Engine:", engine)


Database path: /Users/fiona_spencer/Desk/ML/notebooks/sqlalchemy_demo/demo.db
Engine: Engine(sqlite:///sqlalchemy_demo/demo.db)


## 3) Engine and connection basics

In [6]:

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Hello SQLAlchemy' AS message"))
    row = result.first()
    print(row)
    print("Message:", row.message)


('Hello SQLAlchemy',)
Message: Hello SQLAlchemy



## 4) SQLAlchemy Core: define tables with `MetaData`

Core is SQLAlchemy's SQL expression and table toolkit.


In [7]:

metadata = MetaData()

users = Table(
    "users",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(120), unique=True, nullable=False),
    Column("age", Integer),
    Column("salary", Float),
    Column("is_active", Boolean, default=True),
    Column("created_at", DateTime, default=datetime.utcnow),
)

orders = Table(
    "orders",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("users.id"), nullable=False),
    Column("product", String(100), nullable=False),
    Column("amount", Float, nullable=False),
    Column("notes", Text),
    Column("created_at", DateTime, default=datetime.utcnow),
)

print("Defined tables:")
for table_name in metadata.tables:
    print("-", table_name)


Defined tables:
- users
- orders


## 5) Create tables

In [8]:

metadata.create_all(engine)
print("Tables created.")


Tables created.


## 6) Inspect table columns

In [9]:

for col in users.columns:
    print(col.name, "|", col.type, "| nullable:", col.nullable, "| primary_key:", col.primary_key)


id | INTEGER | nullable: False | primary_key: True
name | VARCHAR(100) | nullable: False | primary_key: False
email | VARCHAR(120) | nullable: False | primary_key: False
age | INTEGER | nullable: True | primary_key: False
salary | FLOAT | nullable: True | primary_key: False
is_active | BOOLEAN | nullable: True | primary_key: False
created_at | DATETIME | nullable: True | primary_key: False


## 7) Insert rows with Core

In [10]:

with engine.begin() as conn:
    conn.execute(insert(users), [
        {"name": "Fiona", "email": "fiona@example.com", "age": 21, "salary": 55000, "is_active": True},
        {"name": "Alex", "email": "alex@example.com", "age": 24, "salary": 62000, "is_active": True},
        {"name": "Sam", "email": "sam@example.com", "age": 19, "salary": 41000, "is_active": False},
        {"name": "Jordan", "email": "jordan@example.com", "age": 28, "salary": 73000, "is_active": True},
    ])
print("Inserted users.")


Inserted users.


In [11]:

with engine.begin() as conn:
    conn.execute(insert(orders), [
        {"user_id": 1, "product": "Laptop", "amount": 1200.0, "notes": "Student discount"},
        {"user_id": 1, "product": "Mouse", "amount": 25.0, "notes": ""},
        {"user_id": 2, "product": "Keyboard", "amount": 80.0, "notes": "Mechanical"},
        {"user_id": 2, "product": "Monitor", "amount": 300.0, "notes": "27 inch"},
        {"user_id": 4, "product": "Desk", "amount": 250.0, "notes": "Wood finish"},
    ])
print("Inserted orders.")


Inserted orders.


## 8) Basic SELECT queries

In [12]:

stmt = select(users)

with engine.connect() as conn:
    result = conn.execute(stmt)
    rows = result.fetchall()

print("All users:")
for row in rows:
    print(row)


All users:
(1, 'Fiona', 'fiona@example.com', 21, 55000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885216))
(2, 'Alex', 'alex@example.com', 24, 62000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))
(3, 'Sam', 'sam@example.com', 19, 41000.0, False, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))
(4, 'Jordan', 'jordan@example.com', 28, 73000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))


In [13]:

stmt = select(users.c.name, users.c.email, users.c.salary)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row)


('Fiona', 'fiona@example.com', 55000.0)
('Alex', 'alex@example.com', 62000.0)
('Sam', 'sam@example.com', 41000.0)
('Jordan', 'jordan@example.com', 73000.0)


## 9) WHERE filtering

In [14]:

stmt = select(users).where(users.c.age >= 21)

with engine.connect() as conn:
    rows = conn.execute(stmt).fetchall()

print("Users age >= 21")
for row in rows:
    print(row)


Users age >= 21
(1, 'Fiona', 'fiona@example.com', 21, 55000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885216))
(2, 'Alex', 'alex@example.com', 24, 62000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))
(4, 'Jordan', 'jordan@example.com', 28, 73000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))


In [15]:

stmt = select(users).where(
    and_(users.c.is_active == True, users.c.salary > 50000)
)

with engine.connect() as conn:
    rows = conn.execute(stmt).fetchall()

print("Active users with salary > 50000")
for row in rows:
    print(row)


Active users with salary > 50000
(1, 'Fiona', 'fiona@example.com', 21, 55000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885216))
(2, 'Alex', 'alex@example.com', 24, 62000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))
(4, 'Jordan', 'jordan@example.com', 28, 73000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))


In [16]:

stmt = select(users).where(
    or_(users.c.name == "Fiona", users.c.name == "Sam")
)

with engine.connect() as conn:
    rows = conn.execute(stmt).fetchall()

print("Users named Fiona or Sam")
for row in rows:
    print(row)


Users named Fiona or Sam
(1, 'Fiona', 'fiona@example.com', 21, 55000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885216))
(3, 'Sam', 'sam@example.com', 19, 41000.0, False, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))


## 10) Ordering, limiting, and offsets

In [17]:

stmt = (
    select(users.c.name, users.c.salary)
    .order_by(desc(users.c.salary))
    .limit(3)
)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row)


('Jordan', 73000.0)
('Alex', 62000.0)
('Fiona', 55000.0)


## 11) Update rows

In [18]:

stmt = (
    update(users)
    .where(users.c.name == "Sam")
    .values(is_active=True, salary=45000)
)

with engine.begin() as conn:
    result = conn.execute(stmt)
    print("Rows updated:", result.rowcount)


Rows updated: 1


In [19]:

with engine.connect() as conn:
    rows = conn.execute(select(users).where(users.c.name == "Sam")).fetchall()

print(rows)


[(3, 'Sam', 'sam@example.com', 19, 45000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885220))]


## 12) Delete rows

In [20]:

stmt = delete(orders).where(orders.c.product == "Mouse")

with engine.begin() as conn:
    result = conn.execute(stmt)
    print("Rows deleted:", result.rowcount)


Rows deleted: 1


## 13) Joins

In [21]:

stmt = (
    select(users.c.name, orders.c.product, orders.c.amount)
    .select_from(users.join(orders, users.c.id == orders.c.user_id))
    .order_by(users.c.name)
)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row)


('Alex', 'Keyboard', 80.0)
('Alex', 'Monitor', 300.0)
('Fiona', 'Laptop', 1200.0)
('Jordan', 'Desk', 250.0)


## 14) Aggregates and GROUP BY

In [22]:

stmt = (
    select(
        users.c.name,
        func.count(orders.c.id).label("num_orders"),
        func.sum(orders.c.amount).label("total_spent")
    )
    .select_from(users.join(orders, users.c.id == orders.c.user_id))
    .group_by(users.c.name)
    .order_by(desc("total_spent"))
)

with engine.connect() as conn:
    rows = conn.execute(stmt).fetchall()

for row in rows:
    print(row)


('Fiona', 1, 1200.0)
('Alex', 2, 380.0)
('Jordan', 1, 250.0)


## 15) Scalar results and helper methods

In [23]:

stmt = select(func.count()).select_from(users)

with engine.connect() as conn:
    count_value = conn.execute(stmt).scalar_one()

print("User count:", count_value)


User count: 4


In [24]:

stmt = select(users).where(users.c.email == "fiona@example.com")

with engine.connect() as conn:
    row = conn.execute(stmt).first()

print("first():", row)


first(): (1, 'Fiona', 'fiona@example.com', 21, 55000.0, True, datetime.datetime(2026, 3, 15, 3, 14, 37, 885216))


## 16) Transactions with `engine.begin()`

In [25]:

with engine.begin() as conn:
    conn.execute(insert(users), {
        "name": "Taylor",
        "email": "taylor@example.com",
        "age": 26,
        "salary": 68000,
        "is_active": True
    })

print("Transaction committed automatically.")


Transaction committed automatically.


## 17) Execute raw SQL with `text()`

In [26]:

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT name, salary FROM users WHERE salary > :min_salary"),
        {"min_salary": 60000}
    )
    rows = result.fetchall()

for row in rows:
    print(row)


('Alex', 62000.0)
('Jordan', 73000.0)
('Taylor', 68000.0)



# Part B — SQLAlchemy ORM

The ORM lets you map Python classes to database tables.


## 18) Define a declarative base

In [27]:

class Base(DeclarativeBase):
    pass


## 19) Define ORM models

In [28]:

class Customer(Base):
    __tablename__ = "customers"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(120), unique=True)
    city: Mapped[str] = mapped_column(String(80))
    created_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.utcnow)

    purchases: Mapped[list["Purchase"]] = relationship(back_populates="customer")

    def __repr__(self):
        return f"Customer(id={self.id}, name={self.name!r}, email={self.email!r}, city={self.city!r})"


class Purchase(Base):
    __tablename__ = "purchases"

    id: Mapped[int] = mapped_column(primary_key=True)
    customer_id: Mapped[int] = mapped_column(ForeignKey("customers.id"))
    item: Mapped[str] = mapped_column(String(100))
    price: Mapped[float] = mapped_column(Float)

    customer: Mapped["Customer"] = relationship(back_populates="purchases")

    def __repr__(self):
        return f"Purchase(id={self.id}, item={self.item!r}, price={self.price})"


## 20) Create ORM tables

In [29]:

Base.metadata.create_all(engine)
print("ORM tables created.")


ORM tables created.


## 21) Create a session

In [30]:

session = Session(engine)
print(session)


## 22) Add ORM objects

In [31]:

c1 = Customer(name="Maya", email="maya@example.com", city="Toronto")
c2 = Customer(name="Noah", email="noah@example.com", city="Montreal")
c3 = Customer(name="Olivia", email="olivia@example.com", city="Toronto")

session.add_all([c1, c2, c3])
session.commit()

print("Inserted customers.")


Inserted customers.


In [32]:

session.add_all([
    Purchase(customer_id=1, item="Notebook", price=8.5),
    Purchase(customer_id=1, item="Backpack", price=45.0),
    Purchase(customer_id=2, item="Headphones", price=120.0),
    Purchase(customer_id=3, item="Water Bottle", price=18.0),
    Purchase(customer_id=3, item="Lamp", price=32.0),
])
session.commit()

print("Inserted purchases.")


Inserted purchases.


## 23) Query ORM objects

In [33]:

customers = session.execute(select(Customer)).scalars().all()

for customer in customers:
    print(customer)


Customer(id=1, name='Maya', email='maya@example.com', city='Toronto')
Customer(id=2, name='Noah', email='noah@example.com', city='Montreal')
Customer(id=3, name='Olivia', email='olivia@example.com', city='Toronto')


In [34]:

toronto_customers = session.execute(
    select(Customer).where(Customer.city == "Toronto")
).scalars().all()

print("Customers in Toronto:")
for customer in toronto_customers:
    print(customer)


Customers in Toronto:
Customer(id=1, name='Maya', email='maya@example.com', city='Toronto')
Customer(id=3, name='Olivia', email='olivia@example.com', city='Toronto')


## 24) Get one object

In [35]:

customer = session.get(Customer, 1)
print("session.get(Customer, 1):", customer)


session.get(Customer, 1): Customer(id=1, name='Maya', email='maya@example.com', city='Toronto')


## 25) Update ORM objects

In [36]:

customer = session.get(Customer, 2)
customer.city = "Vancouver"
session.commit()

print(session.get(Customer, 2))


Customer(id=2, name='Noah', email='noah@example.com', city='Vancouver')


## 26) Delete ORM objects

In [37]:

purchase = session.get(Purchase, 4)
session.delete(purchase)
session.commit()

print("Deleted purchase with id=4")


Deleted purchase with id=4


## 27) Relationships

In [38]:

customer = session.get(Customer, 1)
print("Customer:", customer)
print("Purchases:")
for purchase in customer.purchases:
    print("-", purchase)


Customer: Customer(id=1, name='Maya', email='maya@example.com', city='Toronto')
Purchases:
- Purchase(id=1, item='Notebook', price=8.5)
- Purchase(id=2, item='Backpack', price=45.0)


In [39]:

purchase = session.get(Purchase, 1)
print("Purchase:", purchase)
print("Related customer:", purchase.customer)


Purchase: Purchase(id=1, item='Notebook', price=8.5)
Related customer: Customer(id=1, name='Maya', email='maya@example.com', city='Toronto')


## 28) ORM join queries

In [40]:

stmt = (
    select(Customer.name, Purchase.item, Purchase.price)
    .join(Purchase, Customer.id == Purchase.customer_id)
    .order_by(Customer.name)
)

for row in session.execute(stmt):
    print(row)


('Maya', 'Notebook', 8.5)
('Maya', 'Backpack', 45.0)
('Noah', 'Headphones', 120.0)
('Olivia', 'Lamp', 32.0)


## 29) ORM aggregates

In [41]:

stmt = (
    select(
        Customer.city,
        func.count(Customer.id).label("num_customers")
    )
    .group_by(Customer.city)
    .order_by(Customer.city)
)

for row in session.execute(stmt):
    print(row)


('Toronto', 2)
('Vancouver', 1)


In [42]:

stmt = select(func.avg(Purchase.price))
avg_price = session.execute(stmt).scalar_one()
print("Average purchase price:", avg_price)


Average purchase price: 51.375


## 30) Rollback example

In [43]:

try:
    bad_customer = Customer(name="Duplicate", email="maya@example.com", city="Ottawa")
    session.add(bad_customer)
    session.commit()
except Exception as e:
    print("Commit failed, rolling back.")
    print("Error type:", type(e).__name__)
    session.rollback()


Commit failed, rolling back.
Error type: IntegrityError


## 31) Session helpers

In [44]:

session_methods = [
    "add", "add_all", "commit", "rollback", "delete",
    "get", "execute", "flush", "close"
]

print("Common Session methods:")
for method in session_methods:
    print("-", method)


Common Session methods:
- add
- add_all
- commit
- rollback
- delete
- get
- execute
- flush
- close


## 32) Common SQLAlchemy Core helpers

In [45]:

core_helpers = [
    "create_engine", "MetaData", "Table", "Column",
    "select", "insert", "update", "delete", "text",
    "func", "and_", "or_", "desc", "asc"
]

for item in core_helpers:
    print("-", item)


- create_engine
- MetaData
- Table
- Column
- select
- insert
- update
- delete
- text
- func
- and_
- or_
- desc
- asc


## 33) Common column types

In [46]:

column_types = [
    "Integer", "String", "Text", "Float", "Boolean",
    "DateTime", "ForeignKey"
]

for item in column_types:
    print("-", item)


- Integer
- String
- Text
- Float
- Boolean
- DateTime
- ForeignKey


## 34) Inspect metadata

In [47]:

print("Core tables:")
for table_name, table_obj in metadata.tables.items():
    print(table_name, "->", list(table_obj.columns.keys()))

print("\nORM tables:")
for table_name, table_obj in Base.metadata.tables.items():
    print(table_name, "->", list(table_obj.columns.keys()))


Core tables:
users -> ['id', 'name', 'email', 'age', 'salary', 'is_active', 'created_at']
orders -> ['id', 'user_id', 'product', 'amount', 'notes', 'created_at']

ORM tables:
customers -> ['id', 'name', 'email', 'city', 'created_at']
purchases -> ['id', 'customer_id', 'item', 'price']


## 35) Show generated SQL

In [48]:

stmt = select(users).where(users.c.salary > 50000).order_by(desc(users.c.salary))
print(stmt)


SELECT users.id, users.name, users.email, users.age, users.salary, users.is_active, users.created_at 
FROM users 
WHERE users.salary > :salary_1 ORDER BY users.salary DESC


## 36) Drop tables example

In [49]:

print("To drop tables later, you can use:")
print("metadata.drop_all(engine)")
print("Base.metadata.drop_all(engine)")


To drop tables later, you can use:
metadata.drop_all(engine)
Base.metadata.drop_all(engine)


## 37) Quick reference

In [50]:

reference = {
    "Engine / connection": ["create_engine", "engine.connect", "engine.begin"],
    "Core schema": ["MetaData", "Table", "Column", "ForeignKey"],
    "Core queries": ["select", "insert", "update", "delete", "text", "func"],
    "ORM base": ["DeclarativeBase", "mapped_column", "Mapped"],
    "ORM session": ["Session", "add", "commit", "rollback", "get", "delete"],
    "Relationships": ["relationship", "ForeignKey"],
    "Query helpers": ["where", "order_by", "group_by", "join", "limit", "scalar_one", "first"],
}

for category, items in reference.items():
    print(f"\n{category}:")
    print(", ".join(items))



Engine / connection:
create_engine, engine.connect, engine.begin

Core schema:
MetaData, Table, Column, ForeignKey

Core queries:
select, insert, update, delete, text, func

ORM base:
DeclarativeBase, mapped_column, Mapped

ORM session:
Session, add, commit, rollback, get, delete

Relationships:
relationship, ForeignKey

Query helpers:
where, order_by, group_by, join, limit, scalar_one, first


## 38) Inspect public names

In [51]:

import sqlalchemy
print("Some public names in sqlalchemy:")
print([name for name in dir(sqlalchemy) if not name.startswith("_")][:120])


Some public names in sqlalchemy:
['ARRAY', 'AdaptedConnection', 'Alias', 'AliasedReturnsRows', 'Any', 'AssertionPool', 'AsyncAdaptedQueuePool', 'BIGINT', 'BINARY', 'BLANK_SCHEMA', 'BLOB', 'BOOLEAN', 'BaseDDLElement', 'BaseRow', 'BigInteger', 'BinaryExpression', 'BindParameter', 'BindTyping', 'Boolean', 'BooleanClauseList', 'CHAR', 'CLOB', 'CTE', 'CacheKey', 'Case', 'Cast', 'CheckConstraint', 'ChunkedIteratorResult', 'ClauseElement', 'ClauseList', 'CollectionAggregate', 'Column', 'ColumnClause', 'ColumnCollection', 'ColumnDefault', 'ColumnElement', 'ColumnExpressionArgument', 'ColumnOperators', 'Compiled', 'CompoundSelect', 'Computed', 'Connection', 'Constraint', 'CreateEnginePlugin', 'CursorResult', 'DATE', 'DATETIME', 'DDL', 'DDLElement', 'DECIMAL', 'DOUBLE', 'DOUBLE_PRECISION', 'Date', 'DateTime', 'DefaultClause', 'Delete', 'Dialect', 'Double', 'Engine', 'Enum', 'ExceptionContext', 'Executable', 'ExecutableDDLElement', 'ExecutionContext', 'Exists', 'Extract', 'FLOAT', 'FallbackAsyncA


## 39) Practice exercises

1. Create a new SQLite engine.
2. Define a table with `id`, `title`, and `price`.
3. Insert 3 rows with `insert()`.
4. Select rows where `price > 20`.
5. Update one row with `update()`.
6. Delete one row with `delete()`.
7. Create ORM classes for `Author` and `Book`.
8. Add a relationship between them.
9. Query all authors and print their books.
10. Try a rollback by inserting a duplicate unique value.



## 40) Summary

You now have a notebook that introduces the main SQLAlchemy workflow:

- create an engine and connect to a database
- define tables with Core
- insert, query, update, delete, and join
- use aggregates and transactions
- define ORM classes
- manage data through `Session`
- create relationships
- query Python objects mapped to tables

This is the core path used in many SQLAlchemy applications, APIs, and data projects.
